# Cinnamon — annual fallback series

Sri Lanka produces most of the world's true cinnamon, and it is one of CeyNex's
five tracked products. This source is a **documented fallback**, and the reason
it exists is the most important thing on this page.

Connector: `ceynex/data/connectors/cinnamon.py` (owner: M1 Dinapura).
Covers **2011–2025**.

## ⚠️ What this is not

The project's cinnamon benchmark comes from **Liyanage & Silva (2025)**, a
"Stacked Boost Forest" model reporting **96–98% accuracy** on 2016–2024 domestic
purchasing prices, broken down by location, grade and date.

**We could not obtain that price panel.** It is not public. So CeyNex uses this
annual fallback workbook instead.

> An annual national series and a location/grade/date panel are not the same
> data. A result produced from this workbook is **not a reproduction of the
> published benchmark** and must never be presented as one.

That has to be said plainly whenever the cinnamon number is shown. It is the
single most likely thing for a marker to press on.

In [1]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

import _common as cx

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (10, 5)

## 1. Is the workbook staged?

In [2]:
AGRI = cx.agriculture_raw_dir()
print("agriculture raw dir:", AGRI or "none found")
print()

workbook = cx.status(
    "Cinnamon",
    (AGRI / "cinnamon") if AGRI else None,
    "cinnamon_annual_fallback_2011_2025.xlsx",
    how="curated by M1 from FAOSTAT and Department of Export Agriculture releases.",
)

agriculture raw dir: /ml/CeyNex/ceynex-core/data/raw

Cinnamon
  staged: NO — the analysis cells below will skip.
  expected at: /ml/CeyNex/ceynex-core/data/raw/cinnamon/cinnamon_annual_fallback_2011_2025.xlsx
  how to get it: curated by M1 from FAOSTAT and Department of Export Agriculture releases.


## 2. Two sources, kept deliberately apart

The workbook has two sheets, and the connector **does not merge them**:

| Sheet | `source` | What it is |
|---|---|---|
| `Annual Series` | `FAOSTAT` | international statistics |
| `DEA EAC Series` | `DEA_EAC` | Sri Lanka's Department of Export Agriculture |

They measure overlapping things by different methods. Merging them would
produce a single series with a methodology change hidden inside it. Keeping them
separate means a disagreement stays visible and can be flagged — which is the
same rule CeyNex applies everywhere else two sources overlap.

Every row carries `flag` and `dq_flags` columns so a known-doubtful observation
travels with its doubt attached.

## 3. What reaches `fact_trade`

A narrow slice: `source == "DEA_EAC"`, `metric == "export_volume"`,
`category == "total"`, `unit == "MT"` — converted to kilograms (`× 1000`).

The FAOSTAT sheet stays staged. So does every price observation: the workbook
carries prices, but the agriculture agent reads those from staging rather than
`fact_trade`, because the two sheets' prices are not on a common basis.

## 4. Load

In [3]:
REQUIRED = [
    "year", "metric", "category", "value", "unit",
    "source", "source_file", "source_url", "flag", "dq_flags",
]

data = None
if workbook is not None:
    frames = []
    for sheet in ("Annual Series", "DEA EAC Series"):
        frame = pd.read_excel(workbook, sheet_name=sheet, header=3)
        missing = set(REQUIRED) - set(frame.columns)
        if missing:
            print(f"{sheet}: missing columns {sorted(missing)}")
            continue
        frame = frame[REQUIRED].copy()
        frame["year"] = pd.to_numeric(frame["year"], errors="coerce").astype("Int64")
        frame["value"] = pd.to_numeric(frame["value"], errors="coerce")
        frame["sheet"] = sheet
        frames.append(frame)

    if frames:
        data = pd.concat(frames, ignore_index=True)
        print(f"{len(data):,} observations, {data['year'].min()}-{data['year'].max()}\n")
        print("rows per source and metric:")
        print(data.groupby(["source", "metric"]).size().to_string())
        display(data.head())
else:
    print("skipped — no workbook staged")

skipped — no workbook staged


## 5. What actually gets written

In [4]:
if data is not None:
    written = data[
        (data["source"] == "DEA_EAC")
        & (data["metric"] == "export_volume")
        & (data["category"] == "total")
        & (data["unit"] == "MT")
    ]
    print(f"{len(data):,} staged observations -> {len(written)} fact_trade rows")
    print(f"({100 * (1 - len(written) / len(data)):.0f}% is staged-only, by design)\n")
    display(written[["year", "value", "unit", "flag", "dq_flags"]])
else:
    print("skipped — no data")

skipped — no data


## 6. Where the two sources disagree

In [5]:
if data is not None:
    overlap = (
        data.pivot_table(
            index=["metric", "category", "unit", "year"],
            columns="source",
            values="value",
            aggfunc="mean",
        )
        .dropna(thresh=2)
    )
    if not overlap.empty and overlap.shape[1] >= 2:
        left, right = overlap.columns[:2]
        overlap["difference_pct"] = 100 * (overlap[right] - overlap[left]) / overlap[left]
        display(overlap.round(1))
        print()
        print("CeyNex keeps both rows and records the disagreement as a dq_flag.")
        print("It never averages them — that produces a number neither source")
        print("would defend.")
    else:
        print("no year/metric is covered by both sources in this workbook")
else:
    print("skipped — no data")

skipped — no data


## 7. Export volume over time

In [6]:
if data is not None:
    volumes = data[(data["metric"] == "export_volume") & (data["category"] == "total")]
    if not volumes.empty:
        wide = volumes.pivot_table(index="year", columns="source", values="value", aggfunc="mean")
        ax = wide.plot(marker="o", title="Sri Lanka cinnamon export volume (metric tonnes)")
        ax.set_ylabel("metric tonnes")
        plt.tight_layout()
        plt.show()
        display(wide.round(0))
    else:
        print("no total export_volume rows in this workbook")
else:
    print("skipped — no data")

skipped — no data


## 8. Cross-check against Comtrade

Comtrade carries cinnamon under HS `0906` and is already ingested, so this
comparison runs whether or not the workbook is staged.

In [7]:
facts = cx.comtrade_parquet()
cinnamon = facts[facts["item"] == "cinnamon"]

summary = cinnamon.groupby("year").agg(
    export_value_usd_mn=("export_value_usd", lambda s: s.sum() / 1e6),
    export_volume_mt=("export_volume", lambda s: s.sum() / 1000),
    partners=("partner_iso3", "nunique"),
)
summary["implied_usd_per_kg"] = (
    summary["export_value_usd_mn"] * 1e6 / (summary["export_volume_mt"] * 1000)
)
display(summary.round(2))
print("Comtrade cinnamon, from the ingested parquet. 2018 is absent because")
print("Sri Lanka reported nothing to Comtrade that year — a gap in the source.")

,export_value_usd_mn,export_volume_mt,partners,implied_usd_per_kg
year,,,,
2015,131.97,13548.63,73,9.74
2016,159.09,14687.12,73,10.83
2017,202.47,16615.36,70,12.19
2019,175.92,17171.38,87,10.24
2020,216.34,19620.73,88,11.03
2021,247.34,18812.62,87,13.15
2022,232.35,18297.88,87,12.70
2023,211.11,19676.17,86,10.73
2024,214.43,19545.79,81,10.97


Comtrade cinnamon, from the ingested parquet. 2018 is absent because
Sri Lanka reported nothing to Comtrade that year — a gap in the source.


In [8]:
if data is not None:
    dea = (
        data[
            (data["source"] == "DEA_EAC")
            & (data["metric"] == "export_volume")
            & (data["category"] == "total")
        ]
        .set_index("year")["value"]
        .rename("dea_eac_mt")
    )
    compare = pd.concat([dea, summary["export_volume_mt"].rename("comtrade_mt")], axis=1).dropna()
    if not compare.empty:
        compare["difference_pct"] = 100 * (
            compare["comtrade_mt"] - compare["dea_eac_mt"]
        ) / compare["dea_eac_mt"]
        display(compare.round(1))
    else:
        print("no overlapping years between the workbook and Comtrade")
else:
    print("workbook not staged — Comtrade-only view above")

workbook not staged — Comtrade-only view above
